In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from lightgbm import LGBMRegressor

RAW = Path("..") / "data" / "raw"
PROC = Path("..") / "data" / "processed"
RESULTS = Path("..") / "results"
RESULTS.mkdir(exist_ok=True)

TARGETS = ["Al", "B", "Ca", "Cu", "Fe", "K", "Mg", "Mn", "N", "Na", "P", "S", "Zn"]
SEED = 42

train = pd.read_csv(RAW / "Train.csv")
test = pd.read_csv(RAW / "TestSet.csv")
X_train = pd.read_csv(PROC / "worldclim_train.csv")
X_test = pd.read_csv(PROC / "worldclim_monthly_test.csv")

X_test=X_test.iloc[:, :20]

# The feature columns must match exactly, in the same order, in both files
# assert list(X_train.columns) == list(X_test.columns), "feature columns differ"
# Each feature row must correspond to one raw row
assert len(X_train) == len(train) and len(X_test) == len(test), "row counts differ"

print(X_train.shape, X_test.shape)
print("Test rows with any missing feature:", X_test.isna().any(axis=1).sum())

(44298, 20) (6070, 20)
Test rows with any missing feature: 0


In [3]:
preds = {}
for t in TARGETS:
    mask = train[t].notna()  # use only training rows that have this nutrient
    model = LGBMRegressor(n_estimators=500, learning_rate=0.05,
                          random_state=SEED, verbose=-1)
    model.fit(X_train[mask], train.loc[mask, t])
    preds[t] = model.predict(X_test)
    print(f"{t:3s} trained on {mask.sum():6d} rows")

pred_df = pd.DataFrame(preds)
pred_df.insert(0, "ID", test["ID"].values)

Al  trained on  44296 rows
B   trained on   1909 rows
Ca  trained on  44298 rows
Cu  trained on  44257 rows
Fe  trained on  44257 rows
K   trained on  44298 rows
Mg  trained on  44298 rows
Mn  trained on  44255 rows
N   trained on  44257 rows
Na  trained on   1948 rows
P   trained on   1909 rows
S   trained on   1909 rows
Zn  trained on   1909 rows


In [4]:
for t in TARGETS:
    print(t, "negative predictions:", (pred_df[t] < 0).sum())
    pred_df[t] = pred_df[t].clip(lower=0)

pred_df.describe().round(2)

Al negative predictions: 0
B negative predictions: 0
Ca negative predictions: 0
Cu negative predictions: 0
Fe negative predictions: 0
K negative predictions: 0
Mg negative predictions: 0
Mn negative predictions: 0
N negative predictions: 0
Na negative predictions: 0
P negative predictions: 11
S negative predictions: 0
Zn negative predictions: 0


,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
count,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00,6070.00
mean,818.49,0.13,2521.83,2.44,111.44,301.08,434.76,128.71,1.16,39.93,14.82,11.48,1.18
std,259.08,0.04,2997.23,1.06,31.83,207.82,323.36,43.29,0.73,9.18,8.08,2.45,0.20
min,160.18,0.01,282.64,0.20,36.87,40.67,71.27,24.31,0.25,10.87,0.00,3.74,0.66
25%,616.57,0.10,846.39,1.63,91.80,136.09,191.81,100.94,0.66,35.03,10.67,10.78,1.05
50%,766.81,0.13,1367.18,2.37,109.67,234.74,325.26,127.52,0.89,42.31,12.47,12.32,1.14
75%,1012.25,0.15,2775.54,3.11,129.20,417.44,574.00,157.93,1.39,46.31,17.92,13.21,1.22
max,1751.63,0.33,20236.82,7.26,228.61,1615.48,2102.74,283.17,6.95,92.42,50.18,19.38,1.79


In [5]:
pred_df.to_csv(RESULTS / "predictions_bio_raw.csv", index=False)
print("Saved", RESULTS / "predictions_bio_raw.csv", pred_df.shape)

Saved ..\results\predictions_bio_raw.csv (6070, 14)


In [6]:
compare = pd.DataFrame({
    "train_mean": train[TARGETS].mean(),
    "train_median": train[TARGETS].median(),
    "pred_mean": pred_df[TARGETS].mean(),
    "pred_median": pred_df[TARGETS].median(),
})
compare["mean_ratio"] = compare["pred_mean"] / compare["train_mean"]
compare.round(2)

,train_mean,train_median,pred_mean,pred_median,mean_ratio
Al,801.06,741.02,818.49,766.81,1.02
B,0.07,0.04,0.13,0.13,1.90
Ca,1894.14,996.39,2521.83,1367.18,1.33
Cu,2.25,2.05,2.44,2.37,1.09
Fe,109.73,102.29,111.44,109.67,1.02
K,245.91,176.95,301.08,234.74,1.22
Mg,361.35,235.02,434.76,325.26,1.20
Mn,126.57,119.00,128.71,127.52,1.02
N,0.98,0.77,1.16,0.89,1.19
Na,37.99,33.40,39.93,42.31,1.05


In [7]:
print(train["Depth_cm"].describe())
print(test["Depth_cm"].describe())

count     44298
unique        2
top        0-20
freq      24078
Name: Depth_cm, dtype: object
count     6070
unique       2
top       0-20
freq      3795
Name: Depth_cm, dtype: object
